### Get best model per MAE score

In [2]:
import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()

best_run = None
best_mae = float("inf")
best_model_path = None

# Get all experiments
experiments = client.search_experiments()

# Loop over all experiments and their runs
for exp in experiments:
    runs = client.search_runs(experiment_ids=[exp.experiment_id])
    for run in runs:
        if 'test_mae' in run.data.metrics:
            mae = run.data.metrics['test_mae']
            if mae < best_mae:
                best_mae = mae
                best_run = run
                best_model_path = f"runs:/{run.info.run_id}/model"

# Register the best model
if best_run:
    model_name = "airbnb-price-regression"
    print(f"Best test_mae: {best_mae:.4f} from run {best_run.info.run_id}")
    print(f"Registering model from: {best_model_path}")

    registered_model = mlflow.register_model(model_uri=best_model_path, name=model_name)
    print(f"Model registered as: {model_name} (version {registered_model.version})")

else:
    print("❌ No runs found with 'test_mae' metric.")

✅ Best test_mae: 67.4555 from run e71f3fc38b3f404c8b0f61a2fe9c550e
📦 Registering model from: runs:/e71f3fc38b3f404c8b0f61a2fe9c550e/model
📌 Model registered as: airbnb-price-regression (version 2)


Registered model 'airbnb-price-regression' already exists. Creating a new version of this model...
Created version '2' of model 'airbnb-price-regression'.


### Deploy Model as REST API

In [4]:
import mlflow
from mlflow.tracking import MlflowClient
import subprocess

client = MlflowClient()

# Get all versions of the model named "baseline"
model_name="airbnb-price-regression"
versions = client.search_model_versions(f"name='{model_name}'")
latest_version = max(versions, key=lambda v: int(v.version))
model_uri = f"models:/{model_name}/{latest_version.version}"
port = 8000
command = [
    "mlflow", "models", "serve",
    "--model-uri", model_uri,
    "--port", str(port),
    "--no-conda"
]
subprocess.run(command)
client.set_registered_model_alias(model_name, version=latest_version.version, alias="champion")
print(f"Starting MLflow model server for {model_uri} on http://localhost:{port}")


Traceback (most recent call last):
  File "/home/erwin/anaconda3/envs/airbnb-demo/bin/mlflow", line 8, in <module>
    sys.exit(cli())
  File "/home/erwin/anaconda3/envs/airbnb-demo/lib/python3.10/site-packages/click/core.py", line 1157, in __call__
    return self.main(*args, **kwargs)
  File "/home/erwin/anaconda3/envs/airbnb-demo/lib/python3.10/site-packages/click/core.py", line 1078, in main
    rv = self.invoke(ctx)
  File "/home/erwin/anaconda3/envs/airbnb-demo/lib/python3.10/site-packages/click/core.py", line 1688, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
  File "/home/erwin/anaconda3/envs/airbnb-demo/lib/python3.10/site-packages/click/core.py", line 1688, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
  File "/home/erwin/anaconda3/envs/airbnb-demo/lib/python3.10/site-packages/click/core.py", line 1434, in invoke
    return ctx.invoke(self.callback, **ctx.params)
  File "/home/erwin/anaconda3/envs/airbnb-demo/lib/python3.10/sit

Starting MLflow model server for models:/airbnb-price-regression/2 on http://localhost:8000


Starting MLflow model server for models:/airbnb-price-regression/1 on http://localhost:8000


2025/03/23 18:17:10 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'
2025/03/23 18:17:10 INFO mlflow.pyfunc.backend: === Running command 'exec uvicorn --host 127.0.0.1 --port 8000 --workers 1 mlflow.pyfunc.scoring_server.app:app'
INFO:     Started server process [27908]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
/home/erwin/anaconda3/envs/airbnb-demo/lib/python3.10/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but GradientBoostingRegressor was fitted without feature names
  warnings.warn(


                         name  ...       amenities
0  Triple Room With City View  ...  ['High chair']

[1 rows x 10 columns]
INFO:     127.0.0.1:37720 - "POST /invocations HTTP/1.1" 200 OK


/home/erwin/anaconda3/envs/airbnb-demo/lib/python3.10/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but GradientBoostingRegressor was fitted without feature names
  warnings.warn(


                         name  ...       amenities
0  Triple Room With City View  ...  ['High chair']

[1 rows x 10 columns]
INFO:     127.0.0.1:37722 - "POST /invocations HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [27908]


KeyboardInterrupt: 